# Comparaison de modeles et recherche d'hyperparametres

Ce notebook suit une demarche propre en deux etapes :

1. comparer plusieurs familles de modeles sur le meme decoupage train/test ;
2. lancer une recherche systematique d'hyperparametres sur chaque candidat ;
3. retenir le meilleur modele selon la performance de validation croisee.

L'objectif est de documenter clairement le choix du modele final et de fournir une preuve exploitable pour la checklist.

In [11]:
import os
import sys
from pathlib import Path

# Remonte depuis le CWD jusqu'a trouver la racine du projet (dossier contenant agritech/).
# Fonctionne quel que soit le repertoire de travail lors du lancement du notebook.
_root = Path(os.getcwd())
while not (_root / "agritech").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [12]:
import json

import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, train_test_split

from agritech.config import MLRUNS_DIR
from agritech.models.train import FEATURE_COLUMNS, MODEL_CANDIDATES, TARGET_COLUMN, build_pipeline, load_training_dataset

In [13]:
dataset = load_training_dataset()
features = dataset[FEATURE_COLUMNS]
target = dataset[TARGET_COLUMN]

x_train, x_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
)

dataset.shape, x_train.shape, x_test.shape

((38382, 7), (30705, 6), (7677, 6))

In [14]:
candidate_models = MODEL_CANDIDATES

scoring = {
    'rmse': 'neg_root_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2',
}


def summarize_holdout(pipeline):
    predictions = pipeline.predict(x_test)
    return {
        'rmse': float(np.sqrt(((y_test - predictions) ** 2).mean())),
        'mae': float(np.abs(y_test - predictions).mean()),
        'r2': float(1 - (((y_test - predictions) ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum())),
    }


def log_run(model_name, baseline_metrics, tuned_metrics, best_params):
    mlflow.log_param('model_name', model_name)
    mlflow.log_param('search_type', 'RandomizedSearchCV')
    mlflow.log_param('feature_columns', ','.join(FEATURE_COLUMNS))
    mlflow.log_params(best_params)
    mlflow.log_metric('baseline_rmse', baseline_metrics['rmse'])
    mlflow.log_metric('baseline_mae', baseline_metrics['mae'])
    mlflow.log_metric('baseline_r2', baseline_metrics['r2'])
    mlflow.log_metric('tuned_rmse', tuned_metrics['rmse'])
    mlflow.log_metric('tuned_mae', tuned_metrics['mae'])
    mlflow.log_metric('tuned_r2', tuned_metrics['r2'])

In [15]:
import time

MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
mlflow.set_experiment('agritech-yield-hyperparam-search')

comparison_rows = []
overall_start = time.time()

for model_name, candidate in candidate_models.items():
    model_start = time.time()
    print(f"Debut: {model_name}", flush=True)

    baseline_pipeline = build_pipeline(candidate['estimator_factory']())
    baseline_pipeline.fit(x_train, y_train)
    baseline_metrics = summarize_holdout(baseline_pipeline)

    print(
        f"Recherche hyperparametres: {model_name} | n_iter={candidate['n_iter']} | cv=5",
        flush=True,
    )

    search = RandomizedSearchCV(
        estimator=build_pipeline(candidate['estimator_factory']()),
        param_distributions=candidate['search_space'],
        n_iter=candidate['n_iter'],
        scoring='neg_root_mean_squared_error',
        cv=5,
        random_state=42,
        n_jobs=-1,
        refit=True,
        verbose=2,
    )
    search.fit(x_train, y_train)

    tuned_pipeline = search.best_estimator_
    tuned_metrics = summarize_holdout(tuned_pipeline)

    with mlflow.start_run(run_name=f'{model_name}_random_search'):
        log_run(model_name, baseline_metrics, tuned_metrics, search.best_params_)

    elapsed_model = time.time() - model_start
    print(
        f"Fin: {model_name} | temps={elapsed_model:.1f}s | best_rmse={tuned_metrics['rmse']:.4f}",
        flush=True,
    )

    comparison_rows.append({
        'model_name': model_name,
        'baseline_rmse': baseline_metrics['rmse'],
        'baseline_mae': baseline_metrics['mae'],
        'baseline_r2': baseline_metrics['r2'],
        'tuned_rmse': tuned_metrics['rmse'],
        'tuned_mae': tuned_metrics['mae'],
        'tuned_r2': tuned_metrics['r2'],
        'best_params': search.best_params_,
    })

comparison = pd.DataFrame(comparison_rows).sort_values('tuned_rmse').reset_index(drop=True)
print(f"Termine en {time.time() - overall_start:.1f}s", flush=True)
comparison

Debut: linear_regression
Recherche hyperparametres: linear_regression | n_iter=2 | cv=5
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fin: linear_regression | temps=0.3s | best_rmse=46985.0351
Debut: random_forest
Recherche hyperparametres: random_forest | n_iter=12 | cv=5
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Fin: random_forest | temps=204.8s | best_rmse=8836.3162
Debut: gradient_boosting
Recherche hyperparametres: gradient_boosting | n_iter=12 | cv=5
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Fin: gradient_boosting | temps=11.7s | best_rmse=26632.5115
Termine en 216.7s


,model_name,baseline_rmse,baseline_mae,baseline_r2,tuned_rmse,tuned_mae,tuned_r2,best_params
0,random_forest,7746.376497,2653.085770,0.993677,8836.316182,3346.195562,0.991772,"{'model__n_estimators': 150, 'model__min_sampl..."
1,gradient_boosting,30730.310229,20051.789102,0.900491,26632.511479,17693.492621,0.925260,"{'model__subsample': 0.7, 'model__n_estimators..."
2,linear_regression,46985.008353,33734.023092,0.767380,46985.035086,33734.063364,0.767379,{'model__fit_intercept': False}


In [16]:
best_row = comparison.iloc[0].to_dict()
best_model_name = best_row['model_name']
best_params = best_row['best_params']

best_model = build_pipeline(candidate_models[best_model_name]['estimator_factory']())
best_model.set_params(**best_params)
best_model.fit(x_train, y_train)
best_predictions = best_model.predict(x_test)

best_metrics = {
    'rmse': float(np.sqrt(((y_test - best_predictions) ** 2).mean())),
    'mae': float(np.abs(y_test - best_predictions).mean()),
    'r2': float(1 - (((y_test - best_predictions) ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum())),
}

best_summary = {
    'best_model_name': best_model_name,
    'best_params': best_params,
    'best_metrics': best_metrics,
}

best_summary

{'best_model_name': 'random_forest',
 'best_params': {'model__n_estimators': 150,
  'model__min_samples_split': 2,
  'model__min_samples_leaf': 1,
  'model__max_features': 0.7,
  'model__max_depth': 20,
  'model__bootstrap': False},
 'best_metrics': {'rmse': 8836.316182121916,
  'mae': 3346.195562356818,
  'r2': 0.9917724307229879}}

In [17]:
display_columns = [
    'model_name',
    'baseline_rmse',
    'tuned_rmse',
    'baseline_mae',
    'tuned_mae',
    'baseline_r2',
    'tuned_r2',
    'best_params',
]

comparison[display_columns]

,model_name,baseline_rmse,tuned_rmse,baseline_mae,tuned_mae,baseline_r2,tuned_r2,best_params
0,random_forest,7746.376497,8836.316182,2653.085770,3346.195562,0.993677,0.991772,"{'model__n_estimators': 150, 'model__min_sampl..."
1,gradient_boosting,30730.310229,26632.511479,20051.789102,17693.492621,0.900491,0.925260,"{'model__subsample': 0.7, 'model__n_estimators..."
2,linear_regression,46985.008353,46985.035086,33734.023092,33734.063364,0.767380,0.767379,{'model__fit_intercept': False}


In [18]:
print(json.dumps(best_summary, indent=2, default=str))

best_model

{
  "best_model_name": "random_forest",
  "best_params": {
    "model__n_estimators": 150,
    "model__min_samples_split": 2,
    "model__min_samples_leaf": 1,
    "model__max_features": 0.7,
    "model__max_depth": 20,
    "model__bootstrap": false
  },
  "best_metrics": {
    "rmse": 8836.316182121916,
    "mae": 3346.195562356818,
    "r2": 0.9917724307229879
  }
}


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['Area','Item','Year','average_rain_fall_mm_per_year','pesticides_tonnes', 'avg_temp']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed throug

## Conclusion

Cette version compare plusieurs familles de modeles, lance une recherche systematique d'hyperparametres sur chacune d'elles, puis retient la meilleure configuration finale. C'est la forme la plus defendable pour la checklist et elle reste coherent avec le script d'entraînement du projet.

La suite logique serait de conserver ce notebook comme preuve et d'utiliser le meme principe comme reference pour les futurs ajustements du pipeline.

In [ ]:
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())

exp_name = 'agritech-yield-hyperparam-search'
experiment = mlflow.get_experiment_by_name(exp_name)

if experiment is None:
    print(f"Experiment '{exp_name}' introuvable. Execute d'abord la cellule d'entrainement.")
else:
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        output_format='pandas',
    )

    if runs.empty:
        print(f"Aucun run pour '{exp_name}'.")
    else:
        cols = [
            'run_id',
            'tags.mlflow.runName',
            'start_time',
            'metrics.tuned_rmse',
            'metrics.tuned_mae',
            'metrics.tuned_r2',
            'params.model_name',
        ]
        available_cols = [c for c in cols if c in runs.columns]
        screenshot_table = runs[available_cols].sort_values('start_time', ascending=False).reset_index(drop=True)
        screenshot_table.head(20)

## Preuves attendues: screenshots MLflow

Pour valider la checklist, prends ces captures apres execution des cellules:

1. Vue Experiments avec `agritech-yield-hyperparam-search` et le nombre de runs.
2. Tableau des runs tries par date (capturer les colonnes `run_id`, `tags.mlflow.runName`, `metrics.tuned_rmse`, `metrics.tuned_mae`, `metrics.tuned_r2`).
3. Detail du meilleur run (onglets Params + Metrics).

Commande UI (dans un terminal a la racine du projet):

`py -m mlflow ui --backend-store-uri ./mlruns`

Puis ouvrir: `http://127.0.0.1:5000`.